# MSigDB ORA Saturation Analysis

**Environment:** `clamp-analyses`

For each CLAMPfull model in the saturation grid (`07_saturation`, varying K across 4 data coverage levels and up to 3 seeds), this notebook:

1. Loads the Z matrix (gene loadings per LV).
2. For each LV, selects the top 1% genes by descending loading as the gene list.
3. Runs `enricher()` per LV using MSigDB (v2026.1) as gene set database and model genes as universe.
4. Stores raw `terms_padj`: the minimum p.adjust per MSigDB term across all LVs (no FDR threshold applied here).
5. Saves per-model RDS caches (`_msigdb.rds`) for downstream saturation plotting. FDR thresholds (0.05 / 0.01) and coverage computation are done in `01_bp_saturation_plot.ipynb`.

`pvalueCutoff = 0.05` (not `1`): with no filtering, `enricher()` returns every one of ~35k MSigDB terms per LV (with a verbose `geneID` column), which OOM-killed runs at large K in other tracks. `0.05` is the loosest FDR threshold used downstream and is behavior-preserving for the coverage calculation.

In [ ]:
library(here)
library(dplyr)
library(clusterProfiler)
library(BiocParallel)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/07_saturation")
output_dir <- here("output/03_model_biology/00_archs4/07_saturation_random/00_bp_saturation_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)

## Model grid

Build the full (rs_pct, k_val, seed) grid and filter to combinations where both Z.csv and CLAMPfull_hall.rds exist.

In [ ]:
rs_pcts  <- c(1L, 5L, 10L, 25L)
k_values <- c(86L, 173L, 432L, 864L, 1296L, 1728L)
seeds    <- 1:3

model_grid <- expand.grid(
  rs_pct = rs_pcts,
  k_val  = k_values,
  seed   = seeds,
  stringsAsFactors = FALSE
)

model_grid$subdir <- sprintf(
  "hall_saturation_rs%d_k%d_seed_%d",
  model_grid$rs_pct, model_grid$k_val, model_grid$seed
)

model_grid$z_path <- file.path(
  models_dir, model_grid$subdir, "CLAMPfull_hall", "Z.csv"
)
model_grid$rds_path <- file.path(
  models_dir, model_grid$subdir, "CLAMPfull_hall.rds"
)

model_grid <- model_grid[file.exists(model_grid$z_path), ]
rownames(model_grid) <- NULL

message("Available models: ", nrow(model_grid))
print(model_grid[, c("rs_pct", "k_val", "seed")])

## Load MSigDB gene sets

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

## Helper: run ORA for one model

Returns a list with raw `terms_padj` (minimum p.adjust per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  bp <- BiocParallel::MulticoreParam(workers = n_cores, progressbar = FALSE)
  ora_results <- BiocParallel::bplapply(
    seq_len(n_lvs),
    function(i) {
      genes <- top_genes_per_lv[, i]
      tryCatch(
        clusterProfiler::enricher(
          gene          = genes,
          universe      = universe_genes,
          TERM2GENE     = msig_gmt,
          pAdjustMethod = "BH",
          pvalueCutoff  = 0.05,
          qvalueCutoff  = 1,
          minGSSize     = 10,
          maxGSSize     = 50000
        ),
        error = function(e) NULL
      )
    },
    BPPARAM = bp
  )

  all_dfs <- lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  })
  all_dfs <- Filter(Negate(is.null), all_dfs)

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

## Helper: build results row from ORA output

In [ ]:
build_row <- function(spec, res) {
  data.frame(
    rs_pct           = spec$rs_pct,
    k_val            = spec$k_val,
    seed             = spec$seed,
    n_samples        = res$n_samples,
    n_lvs            = res$n_lvs,
    n_top_genes      = res$n_top_genes,
    n_total_msigdb   = res$n_total_msigdb,
    stringsAsFactors = FALSE
  )
}

## Run ORA: CLAMPfull

In [ ]:
results_list <- lapply(seq_len(nrow(model_grid)), function(i) {
  spec <- model_grid[i, ]

  cache_path <- file.path(
    output_dir, "CLAMPfull",
    sprintf("rs%d_k%d_seed%d_msigdb.rds", spec$rs_pct, spec$k_val, spec$seed)
  )

  if (file.exists(cache_path)) {
    message(sprintf("Loading cached: rs%d k%d seed%d", spec$rs_pct, spec$k_val, spec$seed))
    res <- readRDS(cache_path)
  } else {
    message(sprintf("Running ORA:    rs%d k%d seed%d", spec$rs_pct, spec$k_val, spec$seed))
    res <- run_ora_for_model(spec$z_path)
    if (!is.null(res)) saveRDS(res, cache_path)
  }

  if (is.null(res)) return(NULL)
  build_row(spec, res)
})

results_df <- do.call(rbind, Filter(Negate(is.null), results_list))
rownames(results_df) <- NULL
results_df <- results_df %>%
  dplyr::arrange(rs_pct, k_val, seed)

message("Collected ", nrow(results_df), " rows")
print(results_df)

In [ ]:
write.csv(
  results_df,
  file.path(output_dir, "results_CLAMPfull_msigdb.csv"),
  row.names = FALSE
)
message("Saved results_CLAMPfull_msigdb.csv")